In [1]:
# !pip install pymysql
# !pip install sqlalchemy
# !pip install sqlalchemy pymysql
# !pip install --upgrade sqlalchemy
#!pip show sqlalchemy

In [3]:
import pandas as pd
import numpy as np
import pymysql
from sqlalchemy import create_engine
from datetime import datetime, timedelta
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score
import joblib
import os

# Database connection details
host = 'localhost'
user = 'towdevuser'
password = 'Dev703'
database = 'timelineofwealth'

# Model file path
model_path = 'xgboost_marketcap_model_monthly_returns.pkl'

# Flag to decide if training should be done from scratch
train_from_scratch = False

# Create a database connection
engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}/{database}")

# Logging function
def log(message):
    print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - {message}")

# Step 1: Load Data from Database
log("Loading data from the database...")
query = """
    SELECT * FROM daily_data_s 
    WHERE name IN (
        SELECT ticker FROM stock_universe a 
        WHERE is_nse500 = 1 OR is_bse500 = 1
    )
    ORDER BY name, date DESC;
"""
df = pd.read_sql(query, engine)
log("Data loaded successfully.")

# Step 2: Convert date column to datetime
df['date'] = pd.to_datetime(df['date'])

# Step 3: Create the target variable (1-month ahead market cap)
log("Creating 1-Month ahead market cap target...")
def get_one_month_ahead_market_cap(df):
    # Ensure data is sorted by name and date
    df = df.sort_values(by=['name', 'date'])
    
    # Create a new column for the target date (exactly 1 month ahead)
    df['target_date'] = df['date'] + pd.DateOffset(months=1)
    
    # Self-join to match each row with its corresponding target date
    df_with_target = df.merge(
        df[['name', 'date', 'market_cap']],
        left_on=['name', 'target_date'],
        right_on=['name', 'date'],
        suffixes=('', '_target'),
        how='left'
    )
    
    # Rename the merged market cap column to 'target_market_cap'
    df_with_target.rename(columns={'market_cap_target': 'target_market_cap'}, inplace=True)
    
    # Drop the duplicate 'date' column from the merged result
    df_with_target.drop(columns=['date_target'], inplace=True)
    
    # Drop rows where 'target_market_cap' is missing
    df_with_target = df_with_target.dropna(subset=['target_market_cap'])
    
    return df_with_target

df = get_one_month_ahead_market_cap(df)
log("Target variable created.")

# Verify for TCS on 2023-01-03
# log("Verifying target market cap for TCS on 2023-01-03...")
# verification_row = df[(df['name'] == 'TCS') & (df['date'] == '2023-01-03')]

# if not verification_row.empty:
#     market_cap = verification_row['market_cap'].values[0]
#     target_market_cap = verification_row['target_market_cap'].values[0]
#     print(f"Name: TCS")
#     print(f"Date: 2023-01-03")
#     print(f"Market Cap: {market_cap}")
#     print(f"Target Market Cap (1 month ahead): {target_market_cap}")
# else:
#     log("No data found for TCS on 2023-01-03.")

# Step 4: Feature Engineering (excluding non-numeric and target columns)
log("Preparing features and target...")
features = df.drop(columns=['name', 'rank', 'last_result_date','market_cap', 'sector', 'industry', 'sub_industry', 'target_date'])
features_sorted = features.sort_values(by='date')
target_sorted = features_sorted['target_market_cap']
features_sorted = features_sorted.drop(columns=['target_market_cap', 'date'])

# Step 5: Train-test split while preserving time order
log("Splitting data into train and test sets while preserving time order...")
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(features_sorted, target_sorted, test_size=0.3, shuffle=False)
    
log("Data split completed.")

# Check if the model already exists and the flag for training
if os.path.exists(model_path) and not train_from_scratch:
    log("Loading existing model...")
    best_model = joblib.load(model_path)
    log("Model loaded successfully.")
else:
    # Step 6: Define XGBoost model and GridSearchCV parameters
    log("Training the model with GridSearchCV...")
    xgb_model = XGBRegressor(objective='reg:squarederror', random_state=42)

    # param_grid = {
    #     'n_estimators': [100, 200],
    #     'max_depth': [3, 5],
    #     'learning_rate': [0.01, 0.1],
    #     'subsample': [0.8],
    #     'colsample_bytree': [0.8]
    # }
    # param_grid = {
    #     'n_estimators': [100, 200, 300, 500],
    #     'max_depth': [3, 5, 7, 9],
    #     'learning_rate': [0.01, 0.05, 0.1, 0.2],
    #     'subsample': [0.6, 0.8, 1.0],
    #     'colsample_bytree': [0.6, 0.8, 1.0]
    # }
    # Best Tree
    param_grid = {
        'n_estimators': [500],
        'max_depth': [5],
        'learning_rate': [0.05],
        'subsample': [1],
        'colsample_bytree': [0.8]
    }

    grid_search = GridSearchCV(estimator=xgb_model, param_grid=param_grid, cv=3, scoring='r2', verbose=2, n_jobs=-1)

    # Measure training time
    start_time = datetime.now()
    grid_search.fit(X_train, y_train)
    end_time = datetime.now()
    log(f"Model training completed in {end_time - start_time}.")

    # Best parameters from GridSearchCV
    log(f"Best Parameters: {grid_search.best_params_}")

    # Save the best model
    best_model = grid_search.best_estimator_
    joblib.dump(best_model, model_path)
    log("Best model saved for future use.")

# Step 7: Evaluate the model on the test set
log("Evaluating the model on the test set...")
start_time = datetime.now()
y_pred = best_model.predict(X_test)
end_time = datetime.now()
log(f"Model testing completed in {end_time - start_time}.")

# Calculate performance metrics
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

log(f"Test RMSE: {rmse:.2f}")
log(f"Test R²: {r2:.2f}")

# Optional: Feature Importance
importances = best_model.feature_importances_
feature_names = X_train.columns  # Ensure feature_names match the training data columns

# Ensure lengths match
if len(importances) != len(feature_names):
    log("Mismatch between feature importances and feature names. Adjusting...")
    min_length = min(len(importances), len(feature_names))
    importances = importances[:min_length]
    feature_names = feature_names[:min_length]

# Create DataFrame and display feature importance
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

log("Feature Importances:")
print(feature_importance_df)

2024-12-30 13:00:32 - Loading data from the database...
2024-12-30 13:02:52 - Data loaded successfully.
2024-12-30 13:02:52 - Creating 1-year ahead market cap target...
2024-12-30 13:02:54 - Target variable created.
2024-12-30 13:02:54 - Preparing features and target...
2024-12-30 13:02:55 - Splitting data into train and test sets while preserving time order...
2024-12-30 13:02:55 - Data split completed.
2024-12-30 13:02:55 - Training the model with GridSearchCV...
Fitting 3 folds for each of 576 candidates, totalling 1728 fits
2024-12-30 19:02:09 - Model training completed in 5:59:14.393387.
2024-12-30 19:02:09 - Best Parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 500, 'subsample': 1.0}
2024-12-30 19:02:09 - Best model saved for future use.
2024-12-30 19:02:09 - Evaluating the model on the test set...
2024-12-30 19:02:10 - Model testing completed in 0:00:00.579579.
2024-12-30 19:02:10 - Test RMSE: 33328.87
2024-12-30 19:02:10 - Test R²: 0.

In [9]:
def get_latest_data_and_predict(ticker, date, engine, model):
    log(f"Fetching data for ticker: {ticker} on date: {date}")

    # SQL query to fetch the data for the specified ticker and date
    query = f"""
        SELECT * FROM daily_data_s
        WHERE name = '{ticker}'
        AND date = '{date}';
    """

    # Fetch the data from the database
    latest_data = pd.read_sql(query, engine)

    if latest_data.empty:
        log(f"No data found for ticker: {ticker} on date: {date}")
        return

    # Convert date column to datetime
    latest_data['date'] = pd.to_datetime(latest_data['date'])

    # Extract relevant fields
    latest_row = latest_data.iloc[0]
    latest_date = latest_row['date']
    current_market_cap = latest_row['market_cap']

    log(f"Latest available date for {ticker}: {latest_date.strftime('%Y-%m-%d')}")

    # Prepare the test data by dropping unnecessary columns
    test_data = latest_row.drop(labels=['date', 'name', 'rank', 'last_result_date', 'market_cap', 'sector', 'industry', 'sub_industry'])

    # Convert to DataFrame with a single row
    test_data_df = pd.DataFrame([test_data])

    # Loading existing model...
    log("Loading existing model...")
    best_model = joblib.load(model_path)
    log("Model loaded successfully.")

    # Predict the target market cap
    predicted_market_cap = model.predict(test_data_df)[0]

    # Calculate expected percentage change
    expected_change = ((predicted_market_cap / current_market_cap) - 1) * 100

    # Print the results
    print(f"\nPrediction Results for {ticker}")
    print(f"Latest Available Date: {latest_date.strftime('%Y-%m-%d')}")
    print(f"Current Market Cap: {current_market_cap:,.2f}")
    print(f"Predicted Target Market Cap (1 month ahead): {predicted_market_cap:,.2f}")
    print(f"Expected % Change Over the Month: {expected_change:.2f}%")

# Example call for TARSONS with a specific date
ticker = 'TCS'
date = '2024-11-27'  # Replace with the desired date in 'YYYY-MM-DD' format
get_latest_data_and_predict(ticker, date, engine, best_model)


2024-12-30 20:51:27 - Fetching data for ticker: TCS on date: 2024-11-27
2024-12-30 20:51:27 - Latest available date for TCS: 2024-11-27
2024-12-30 20:51:27 - Loading existing model...
2024-12-30 20:51:27 - Model loaded successfully.

Prediction Results for TCS
Latest Available Date: 2024-11-27
Current Market Cap: 1,574,844.95
Predicted Target Market Cap (1 year ahead): 1,224,773.38
Expected % Change Over the Year: -22.23%


In [11]:
import pandas as pd
import joblib
from sqlalchemy import create_engine
import datetime

def log(message):
    print(f"[LOG] {message}")

# Database connection details
host = 'localhost'
user = 'towdevuser'
password = 'Dev703'
database = 'timelineofwealth'

# Load the model
model_path = 'C:/Users/Sudhir Kulaye/JupyterProjects/Models/xgboost_marketcap_model.pkl'  
best_model = joblib.load(model_path)

# Database connection setup
engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}/{database}")

# Function to fetch max date and 1 month prior date
def get_dates(engine):
    query = """
        SELECT MAX(date) AS max_date
        FROM daily_data_s;
    """
    result = pd.read_sql(query, engine)
    max_date = result['max_date'][0]
    
    # Calculate 1 month before the max date
    date_1mth_before = max_date - datetime.timedelta(days=30)
    
    # Find the closest available date less than or equal to date_1mth_before
    query = f"""
        SELECT MAX(date) AS max_date_1mth
        FROM daily_data_s
        WHERE date <= '{date_1mth_before}';
    """
    result_1mth = pd.read_sql(query, engine)
    max_date_1mth = result_1mth['max_date_1mth'][0]
    
    return max_date, max_date_1mth

# Function to get unique stock names for a specific date
def get_unique_stocks(date, engine):
    query = f"""
        SELECT DISTINCT name
        FROM daily_data_s
        WHERE date = '{date}';
    """
    stocks = pd.read_sql(query, engine)
    return stocks['name'].tolist()

# Function to get sector and industry details for a ticker
def get_sector_and_industry(ticker, engine):
    query = f"""
        SELECT a.ticker, b.sector_name_display, b.industry_name_display
        FROM stock_universe a
        JOIN subindustry b ON a.subindustryid = b.subindustryid
        WHERE a.ticker = '{ticker}';
    """
    result = pd.read_sql(query, engine)
    if not result.empty:
        return result.iloc[0]['sector_name_display'], result.iloc[0]['industry_name_display']
    else:
        return '', ''

# Prediction function
def get_latest_data_and_predict(ticker, date, engine, model):
    log(f"Fetching data for ticker: {ticker} on date: {date}")
    
    query = f"""
        SELECT * FROM daily_data_s
        WHERE name = '{ticker}'
        AND date = '{date}';
    """
    
    latest_data = pd.read_sql(query, engine)
    
    if latest_data.empty:
        log(f"No data found for ticker: {ticker} on date: {date}")
        return None
    
    latest_row = latest_data.iloc[0]
    latest_date = latest_row['date']
    current_market_cap = latest_row['market_cap']
    
    test_data = latest_row.drop(labels=['date', 'name', 'rank', 'last_result_date', 'market_cap', 'sector', 'industry', 'sub_industry'])
    test_data_df = pd.DataFrame([test_data])
    
    predicted_market_cap = model.predict(test_data_df)[0]
    expected_change = (predicted_market_cap / current_market_cap) - 1
    
    return {
        'ticker': ticker,
        'date': latest_date,
        'mcap': round(current_market_cap, 2),
        'mcap_predict': round(predicted_market_cap, 2),
        'expected_change': round(expected_change, 4)
    }

# Main function to run predictions for all stocks and compile results
def main():
    max_date, max_date_1mth = get_dates(engine)
    stocks = get_unique_stocks(max_date, engine)
    
    results = []
    
    for stock in stocks:
        sector, industry = get_sector_and_industry(stock, engine)
        
        # Prediction for 1 month before
        result_1mth = get_latest_data_and_predict(stock, max_date_1mth, engine, best_model)
        
        # Prediction for today (max date)
        result_today = get_latest_data_and_predict(stock, max_date, engine, best_model)
        
        if result_today:
            if result_1mth:
                actual_mcap_change = round((result_today['mcap'] / result_1mth['mcap']) - 1, 4)
                diff = round(result_1mth['expected_change'] - actual_mcap_change, 4)
                mcapY0E_percent = round((result_today['mcap_predict'] / result_today['mcap']) - 1, 4)
                
                results.append({
                    'ticker': stock,
                    'sector': sector,
                    'industry': industry,
                    'date_Y1': result_1mth['date'],
                    'mcap_Y1': result_1mth['mcap'],
                    'mcap_Y1E': result_1mth['mcap_predict'],
                    'mcap_Y1E%': result_1mth['expected_change'],
                    'mcap_Y1A%': actual_mcap_change,
                    'diff': diff,
                    'date_Y0': result_today['date'],
                    'mcap_Y0': result_today['mcap'],
                    'mcap_Y0E': result_today['mcap_predict'],
                    'mcapY0E%': mcapY0E_percent
                })
            else:
                results.append({
                    'ticker': stock,
                    'sector': sector,
                    'industry': industry,
                    'date_Y1': '',
                    'mcap_Y1': '',
                    'mcap_Y1E': '',
                    'mcap_Y1E%': '',
                    'mcap_Y1A%': '',
                    'diff': '',
                    'date_Y0': result_today['date'],
                    'mcap_Y0': result_today['mcap'],
                    'mcap_Y0E': result_today['mcap_predict'],
                    'mcapY0E%': ''
                })
    
    # Convert results to DataFrame and save to CSV
    results_df = pd.DataFrame(results)
    output_path = f"{max_date.strftime('%Y%m%d')}_stock_mcap_monthly_predictions.csv"
    results_df.to_csv(output_path, index=False)
    log(f"Results saved to {output_path}")

if __name__ == "__main__":
    main()

[LOG] Fetching data for ticker: 360ONE on date: 2024-11-27
[LOG] Fetching data for ticker: 360ONE on date: 2024-12-27
[LOG] Fetching data for ticker: 3MINDIA on date: 2024-11-27
[LOG] Fetching data for ticker: 3MINDIA on date: 2024-12-27
[LOG] Fetching data for ticker: 508486 on date: 2024-11-27
[LOG] Fetching data for ticker: 508486 on date: 2024-12-27
[LOG] Fetching data for ticker: 532468 on date: 2024-11-27
[LOG] Fetching data for ticker: 532468 on date: 2024-12-27
[LOG] Fetching data for ticker: AADHARHFC on date: 2024-11-27
[LOG] Fetching data for ticker: AADHARHFC on date: 2024-12-27
[LOG] Fetching data for ticker: AARTIDRUGS on date: 2024-11-27
[LOG] Fetching data for ticker: AARTIDRUGS on date: 2024-12-27
[LOG] Fetching data for ticker: AARTIIND on date: 2024-11-27
[LOG] Fetching data for ticker: AARTIIND on date: 2024-12-27
[LOG] Fetching data for ticker: AAVAS on date: 2024-11-27
[LOG] Fetching data for ticker: AAVAS on date: 2024-12-27
[LOG] Fetching data for ticker: ABB on